In [1]:
from BERT import mlm_custom_loss, nsp_custom_loss
from BERT import get_bert_model
import numpy as np
import tensorflow as tf

DISERT_DATA_PATH = r'C:\Users\yunus\Desktop\DisERT Data'
CHECKPOINT_PATH = r'C:\Users\yunus\Desktop\Checkpoints\DisERT'
sCheckPointFilePath = f'{CHECKPOINT_PATH}\model'

X_MLM = np.load(f'{DISERT_DATA_PATH}\X_MLM.npy')
Y_MLM = np.load(f'{DISERT_DATA_PATH}\Y_MLM.npy')
X_NSP = np.load(f'{DISERT_DATA_PATH}\X_NSP.npy')
Y_NSP = np.load(f'{DISERT_DATA_PATH}\Y_NSP.npy')

X_MLM = X_MLM[:4000]
X_NSP = X_NSP[:4000]
Y_MLM = Y_MLM[:4000]
Y_NSP = Y_NSP[:4000]

In [2]:
import time
import os

class CsvLogger(tf.keras.callbacks.Callback):
    
    def __init__(self, sFilePath, sDelimeter = ';'):
        sFilePath = f'{sFilePath}.txt'
        self.sFilePath = sFilePath
        self.sDelimeter = sDelimeter


    def on_epoch_begin(self, batch, logs={}):
        self.epoch_time_start = time.time()
        

    def on_epoch_end(self, epoch, logs={}):
        endTime = time.time()
        
        logs['start'] = self.epoch_time_start
        logs['end'] = endTime
        logs['duration'] = endTime - self.epoch_time_start
        
        logs['epoch'] = epoch
        logs['learning_rate'] = self.model.optimizer.lr.numpy()
        
        if os.path.exists(self.sFilePath) == False:
            with open(self.sFilePath, 'a') as f: 
                f.write(self.sDelimeter.join([str(i) for i in logs.keys()]))
                f.write('\n') 
                
        with open(self.sFilePath, 'a') as f: 
            f.write(self.sDelimeter.join([str(i) for i in logs.values()]))
            f.write('\n')
            
    
class ThresholdStopper(tf.keras.callbacks.Callback):
    def __init__(self, threshold):
        self.threshold = threshold

    def on_batch_end(self, batch, logs={}):
        if logs.get('loss') <= self.threshold:
             self.model.stop_training = True
                
                
sCsvLogFilePath = f'{CHECKPOINT_PATH}\log'
oCsvLogger=  CsvLogger(sCsvLogFilePath)

oDisERTCheckPoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=sCheckPointFilePath,
    save_weights_only=False,
    monitor='loss',
    mode='min',
    save_best_only=True,
    save_traces = True
)

oLearningRateReducer = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='loss', 
    factor=0.80,
    patience=3, 
    min_lr=1e-4
)

oEarlyStopper = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=20)

oThresholdStopper = ThresholdStopper(0.05)

In [3]:
# oDisERT = get_bert_model(
#     mlm_input_shape = (10, 129), 
#     nsp_input_shape = (1, 129),
#     nr_of_encoder_blocks = 4,
#     attention_key_dims = 32,
#     attention_nr_of_heads = 2,
#     attention_dense_dims = 128,
#     dropout_rate = 0.0
# )

# oDisERT.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
#     loss = [mlm_custom_loss, nsp_custom_loss]
# )

# oDisERT.fit(
#     x = [X_MLM, X_NSP], 
#     y = [Y_MLM, Y_NSP], 
#     batch_size= 512,
#     epochs=2,
#     verbose=1,
#     # callbacks = [oDisERTCheckPoint, oCsvLogger, oLearningRateReducer, oEarlyStopper, oThresholdStopper]
# )


# oDisERT.save(sCheckPointFilePath)

In [4]:
# del oDisERT_2
oDisERT_2 = tf.keras.models.load_model(
    sCheckPointFilePath, 
    custom_objects = {
        # 'PositionEncoder':PositionEncoder,
        # 'TransformerEncoder': TransformerEncoder,
        'mlm_custom_loss':mlm_custom_loss,
        'nsp_custom_loss':nsp_custom_loss
    }
)

oDisERT_2.fit(
    x = [X_MLM, X_NSP], 
    y = [Y_MLM, Y_NSP], 
    batch_size= 512,
    epochs=10,
    verbose=1,
    callbacks = [oLearningRateReducer]
    # callbacks = [oDisERTCheckPoint, oCsvLogger, oLearningRateReducer, oEarlyStopper, oThresholdStopper]
)

oDisERT_2.save(sCheckPointFilePath)

Epoch 1/10
8/8 [==============================] - 12s 353ms/step - loss: 0.6352 - mlm_classifier_loss: 0.0426 - nsp_classifier_loss: 0.5926 - lr: 8.0000e-04
Epoch 2/10
8/8 [==============================] - 3s 397ms/step - loss: 0.5776 - mlm_classifier_loss: 0.0412 - nsp_classifier_loss: 0.5363 - lr: 8.0000e-04
Epoch 3/10
8/8 [==============================] - 3s 390ms/step - loss: 0.5032 - mlm_classifier_loss: 0.0410 - nsp_classifier_loss: 0.4622 - lr: 8.0000e-04
Epoch 4/10
8/8 [==============================] - 3s 368ms/step - loss: 0.4571 - mlm_classifier_loss: 0.0407 - nsp_classifier_loss: 0.4164 - lr: 8.0000e-04
Epoch 5/10
8/8 [==============================] - 3s 371ms/step - loss: 0.4120 - mlm_classifier_loss: 0.0399 - nsp_classifier_loss: 0.3721 - lr: 8.0000e-04
Epoch 6/10
8/8 [==============================] - 3s 364ms/step - loss: 0.3796 - mlm_classifier_loss: 0.0395 - nsp_classifier_loss: 0.3402 - lr: 8.0000e-04
Epoch 7/10
8/8 [==============================] - 3s 357ms/step

INFO:tensorflow:Assets written to: C:\Users\yunus\Desktop\Checkpoints\DisERT\model\assets


INFO:tensorflow:Assets written to: C:\Users\yunus\Desktop\Checkpoints\DisERT\model\assets
